# Project_Paru_CXR - Runner Kaggle
**Pengaturan notebook Kaggle:** Accelerator = *GPU T4 x2* (atau P100), Internet = **On**, Persistence = *Files only*.

**Add Data (panel kanan):**
1. Kompetisi `rsna-pneumonia-detection-challenge` (klik *Join Competition* dulu di halaman kompetisi).
2. Dataset `paultimothymooney/chest-xray-pneumonia` (untuk S6 uji eksternal).

Urutan: persiapan data → EDA → split → smoke test → eksperimen → rekap → Grad-CAM.
Setiap tahap yang sudah selesai akan dilewati otomatis bila dijalankan ulang.

In [ ]:
!git clone https://github.com/monicabortang/Project_Paru_CXR.git 2>/dev/null || (cd Project_Paru_CXR && git pull)
%cd /kaggle/working/Project_Paru_CXR
!pip install -q timm pydicom gradio
import torch, timm; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))

In [ ]:
RSNA = '/kaggle/input/rsna-pneumonia-detection-challenge'
DATA = '/kaggle/working/data'
SPLITS = '/kaggle/working/Project_Paru_CXR/splits'  # split dari laptop (di-commit ke GitHub) agar identik
KERMANY_SRC = '/kaggle/input/chest-xray-pneumonia/chest_xray'
EXT = '/kaggle/working/external/kermany'

## 1. Konversi DICOM → PNG 512 px + metadata (±15-25 menit, sekali saja)
Setelah selesai, simpan folder `data/` sebagai Kaggle Dataset agar tidak perlu mengulang.

In [ ]:
import os
if not os.path.exists(f'{DATA}/metadata.csv'):
    !python -m src.prepare_data --rsna_dir {RSNA} --out_dir {DATA} --size 512 --workers 4

## 2. EDA (luaran Progress 2)

In [ ]:
!python -m src.eda --meta {DATA}/metadata.csv --img_root {DATA} --out results/eda
from IPython.display import Image, display
for f in ['grafik_kelas_per_view.png','grafik_usia_per_view.png','contoh_citra.png']:
    display(Image(f'results/eda/{f}'))

## 3. Pembagian data (seed 42)
Split yang dibuat di laptop (folder `splits/` di repositori) dipakai langsung agar identik dengan laporan Progress 2. Nama file citra (`<patientId>.png`) sama, jadi CSV split cocok dengan data hasil konversi di Kaggle.

In [ ]:
import os
if not os.path.exists(f'{SPLITS}/summary.csv'):
    !python -m src.make_splits --meta {DATA}/metadata.csv --out {SPLITS} --seed 42
else:
    print('Memakai split dari repositori:', SPLITS)

## 4. Data uji eksternal (S6) - Kermany

In [ ]:
if not os.path.exists(f'{EXT}/kermany.csv'):
    !python -m src.external kermany --src {KERMANY_SRC} --out {EXT} --subsets test

## 5. Smoke test (± 5 menit) - pastikan pipeline jalan sebelum run panjang

In [ ]:
!python scripts/run_experiments.py --group smoke --splits {SPLITS} --img_root {DATA}

## 6. Eksperimen inti (S1, S2-matched, S3-matched × 2 model × 3 seed)
Bisa dipecah ke beberapa sesi Kaggle; run yang sudah selesai dilewati. Gunakan `--models efficientnet_b0` untuk Progress 3 (baseline) dan `--models swin_tiny_patch4_window7_224` untuk Progress 4.

In [ ]:
EXTARG = f'--external kermany={EXT}/kermany.csv'
!python scripts/run_experiments.py --group core --splits {SPLITS} --img_root {DATA} {EXTARG}

## 7. Skenario tambahan (natural, S4, ablasi)

In [ ]:
!python scripts/run_experiments.py --group natural  --splits {SPLITS} --img_root {DATA}
!python scripts/run_experiments.py --group s4       --splits {SPLITS} --img_root {DATA}
!python scripts/run_experiments.py --group ablation --splits {SPLITS} --img_root {DATA} {EXTARG}

## 8. Rekap tabel

In [ ]:
!python -m src.summarize --results results/all_results.csv --runs runs --out results/tables
import pandas as pd
display(pd.read_csv('results/tables/tabel_domain_shift.csv'))
display(pd.read_csv('results/tables/tabel_uji_berpasangan.csv'))

## 9. Grad-CAM vs bounding box

In [ ]:
for m in ['efficientnet_b0', 'swin_tiny_patch4_window7_224']:
    for scen, test in [('S1_all','all'), ('S2_AP_matched','PA_matched'), ('S3_PA_matched','AP_matched')]:
        rd = f'runs/{scen}/{m}/seed0'
        !python -m src.gradcam --run {rd} --test_csv {rd}/preds_{test}.csv --img_root {DATA} --max_images 500

## 10. Simpan hasil (unduh dari tab Output, lalu commit ke GitHub tanpa best.pt besar)

In [ ]:
!zip -qr /kaggle/working/hasil_eksperimen.zip results runs -x '*.pt'
!du -sh runs results